#### 임베딩

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_classic import hub 
import getpass
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_classic.chains import RetrievalQA 



load_dotenv()  # 환경변수 생성 
embedding = OpenAIEmbeddings(model='text-embedding-3-large')  # 디폴트 모델 : text-embedding-ada-002 -> 변경할 것

index_name = "tax-markdown-index"  # change if desired
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

llm = ChatOpenAI(model='gpt-4o')

prompt = hub.pull("rlm/rag-prompt")

retriever = database.as_retriever(search_kwargs={"k": 4})  # retriever는 데이터베이스에서 정보를 가져오는 역할
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type_kwargs={"prompt" : prompt}
)

ai_message= qa_chain({"query" : query})

dictionary = ["사람을 나타내는 표현 -> 거주자"]
prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전 : {dictionary}
    사용자의 질문 : {{question}}
                                          
""")


dictionary_chain = prompt|llm|StrOutputParser()
tax_chain = {"query" : dictionary_chain} | qa_chain  